In [ ]:
# This notebook requires GPU
# The first part of this notebook is similar to Class4-1, so let's go straight into running
# all the chunks until and including the one where we plot the results

# read in the dataset 

# replace the argument below with the filepath that we have copied 
# we use the open() function to open the file, returning a file object
# subsequently, we use the read() method to read from the file, saving the info into "data"
data = open('/kaggle/input/english-poems-dataset2/poems.txt').read()

In [ ]:
# if there is an error when importing keras-nlp later
# 1. turn off the session
# 2. comment out this line, and re-run 
# !pip install tf_keras

In [ ]:
# import the necessary libraries

import tensorflow as tf
from tensorflow import keras
import numpy as np

In [ ]:
# we look at the first 100 characters and notice there's a newline character \n
data[:100]

In [ ]:
# Let's do some data preparation
# We'll convert all to lowercase and split according to a new line character, using the respective methods
# save into the list "orig_fulltext"

raw_orig_fulltext = data.lower().split("\n")

# check the number of lines in our list
len(raw_orig_fulltext)

# Does the number of lines comport with what we expect? 

In [ ]:
# we look at the first 10 lines
raw_orig_fulltext[:10]

In [ ]:
# we notice that there are empty strings amongst the 10 lines we sliced
# let's filter out all the empty strings from our list

# define an empty list to contain the filtered lines 
orig_fulltext = []

# go through each line, and if it's not an empty string, append to the new list
for x in raw_orig_fulltext: 
    if x != '': 
        orig_fulltext.append(x)

In [ ]:
# we check the first l0 lines again
orig_fulltext[:10]

In [ ]:
# we see how many lines in our filtered list
len(orig_fulltext)

In [ ]:
# we remove punctuation
import string

# create an empty list to store the lines sans punctuation
fulltext = []

# going through each line
for line in orig_fulltext:
    # using the translate() method to remove punctuation, by way of the mapping table  
    # defined using maketrans()
    line = line.translate(str.maketrans('', '', string.punctuation))
    fulltext.append(line)

fulltext[:10]

In [ ]:
# we will use the Tokenizer for text vectorization, as what we learned in our earlier class
from tensorflow.keras.preprocessing.text import Tokenizer

# specify the OOV token; the OOV token will then be added to word_index and used to replace 
# out-of-vocabulary words during text_to_sequence calls 
oov_tok = ""

# instantiate our Tokenizer
tokenizer = Tokenizer(oov_token=oov_tok, split=" ", char_level = False)

# learn the vocab
tokenizer.fit_on_texts(fulltext)

# set the number of words in our vocabulary to the number in our word index + 1
total_words = len(tokenizer.word_index) + 1

# check how many words we have in our vocab
print(total_words)

In [ ]:
# formally set the vocab size to the total number of words
vocab_size = total_words

In [ ]:

# why do we need to add 1 to the length of tokenizer.word_index to get the vocab_size? 
# our tokenizer.word_index is a Python dictionary of key:value pairs, starting with 
# key = 1 for the first token, the OOV
print(tokenizer.word_index)

# we see below that the last key value pair is 3080: 'nay', meaning we have 3080 words in our vocab
# Let's continue the discussion in our slides

In [ ]:
# this is the first line, containing 8 words
fulltext[0]

In [ ]:
# vectorize each line

input_sequences = tokenizer.texts_to_sequences(fulltext)

# take a look at the first 20 members of our list
input_sequences[:20]
# indeed we see that we tokenize each line

In [ ]:
# we need to create features and labels and pre-pad them to the same length <let's go to our slides>

# each feature would be the vectorized sentence up till and excluding the last word
# the label would be the vectorized sentence from the second word onwards

# creating an empty list each for our features and labels
features = []
labels = []

# we cycle through each vectorised sequence
for line in input_sequences: 
    # extract the part sans the last word, and append to features
    feature = line[:-1]
    features.append(feature)
    # extract the part sans the first word, and append to features
    label = line[1:]
    labels.append(label)

In [ ]:
# checking the results
features[0]

In [ ]:
labels[0]

In [ ]:
# because the sequences are all of different lengths, we pre-pad the sequences
from tensorflow.keras.preprocessing.sequence import pad_sequences

# we get the longest vectorized line in our dataset
max_sequence_length = max([len(x) for x in input_sequences])

# we see that it is 62 words long
max_sequence_length

In [ ]:
# since features and labels are one character less, we will pre-pad up to 61 tokens
features = np.array(pad_sequences(features, maxlen=max_sequence_length - 1, padding='pre'))
labels = np.array(pad_sequences(labels, maxlen=max_sequence_length - 1, padding='pre'))

In [ ]:
# this is the first padded feature
features[0]

In [ ]:
# this is the first padded label
labels[0]

In [ ]:
# check the shape for our features array
features.shape

In [ ]:
# check the shape for our labels array
labels.shape

In [ ]:
# install the library that we need for transformers

# requires keras 3
# !pip install keras-nlp --upgrade
!pip install keras-nlp

In [ ]:
# do the necessary imports
import keras_nlp
from keras import initializers
from tensorflow.keras import layers

In [ ]:
# we build the model, using the same parameters as our last class

embed_dim = 64
layer_dim = 16
num_heads = 2

# we instantiate the decoder; 
# the TransformerDecoder class will always apply causal masking to the decoder attention layer
# so we don't have to explicitly set it using an argument, unlike the padding mask 
# why (only) the decoder in this case? Let's go to our slides
decoder = keras_nlp.layers.TransformerDecoder(intermediate_dim = layer_dim, num_heads = num_heads, 
                                             kernel_initializer = initializers.glorot_uniform(seed = 0))

# our inputs are 61-dimensional padded feature vectors
inputs = keras.Input(shape=((max_sequence_length - 1),), dtype="int64")
# for each word in our vocab, we generate a 64-dimensional embedding vector
# note the padding mask applied
x = layers.Embedding(vocab_size, embed_dim, mask_zero = True)(inputs)

# we add the positional encoding vectors to our word vectors, to get new position-aware embedding vectors
# to do this, we generate the position encoding layer first using the SinePositionEncoding class
position_encoding = keras_nlp.layers.SinePositionEncoding()(x)
# then we add the position encoding layer to our embedding layer 
new_position_encoding = x + position_encoding

# we pass the position-aware embedding vectors through the decoder
x = decoder(new_position_encoding)

# code for later; comment out first
# x = decoder(x)

# Let's go to our slides to talk more on the output layer, but let's run this code chunk first
outputs = layers.Dense(vocab_size, activation="softmax")(x)


model = keras.Model(inputs, outputs)

# set our model for training
# similar to our choice of activation function above, "sparse categorical crossentropy" is just 
# an extension of "binary crossentropy", but for multi-class classification
model.compile(loss="sparse_categorical_crossentropy", optimizer = 'rmsprop', metrics = ['accuracy'])

# we train for 150 epochs
num_epochs = 150

# we save the training history into a variable so we can plot the results subsequently
history = model.fit(features, labels, epochs = num_epochs, verbose = 1)

In [ ]:
import matplotlib.pyplot as plt

def plot_graphs(process, metric, filename):      
    plt.plot(process.history[metric])
    plt.title(filename.split('.')[0])
    plt.xlabel("Epochs")
    plt.ylabel(metric)
    plt.savefig(filename)
    plt.show()

In [ ]:
# plotting the results
plot_graphs(history, "accuracy", "Poems - Transformer Accuracy.png")
plot_graphs(history, "loss", "Poems - Transformer Loss.png")

# What do we think about the results? Let's go to the slides to discuss 

In [ ]:
# let's test it on an existing sentence (that is part of the training set)
# we will see if it is able to predict "love", the last word based on the preceding 
# sentence "Or like the poetess's heart of"
seed_text = "Or like the poetess's heart of"

# we tokenize this sentence, and extracting the one list of tokens that we have by indexing [0]
token_list = tokenizer.texts_to_sequences([seed_text])[0]
token_list

In [ ]:
# we now need to pad this to get it into the same shape as the data used for training
# since our features were one token less than the max sequence length, we have use max_sequence_length - 1
token_list = pad_sequences([token_list], maxlen = max_sequence_length - 1, padding = 'pre')
token_list

In [ ]:
# probability distribution given by softmax
predicted_prob = model.predict(token_list)

predicted_prob

In [ ]:
# the data structure is a numpy array
type(predicted_prob)

In [ ]:
predicted_prob.shape

# the shape is 1, 61, 3081
# we have one sentence of 61 words to predict as the outcome (remember that the predicted length (label)
# is the same as the input feature)
# each word in this one sentence can take on one of 3081 words (vocab size)
# so we have 3081 probabilities for each of these words (including the mask token)

In [ ]:
np.sum(predicted_prob, axis = 2)
# we see that the 3081 values of probabilities for each token sum to ~1, as we expect

In [ ]:
# convert to an array of float64 values, so we can store higher precision later as we apply
# the temperature calculations. The higher precision allows us to better check, when we 
# sum the probabilities, if they do indeed go to 1
predicted_prob = np.asarray(predicted_prob).astype('float64')
predicted_prob 

In [ ]:
# now we moderate the probability distribution using softmax temperature
# we set the temperature value: higher temperatures will have more randomness
# you can try the following values: 0.2, 0.5, 0.7, 1.0, 1.5, 2.0
temperature = 1.5

# apply the temperature to the original probability distribution, per the steps described in our slides earlier
predicted_prob = np.log(predicted_prob) / temperature
exp_predicted_prob = np.exp(predicted_prob)
# the arguments in the sum() method is to specify that we are summing along axis = 2, where the probabilities 
# for each word position in the label are located
# keepdims = 1 or True ensures that we keep this dimension even after summing
# dividing each probability with the total ensures that the sum of probabilities now goes to 1
new_predicted_prob = exp_predicted_prob / exp_predicted_prob.sum(axis = 2, keepdims = 1)
new_predicted_prob

# Let's go to our slides to visualise the code 

In [ ]:
# check that for each word, the sum of the new probabilities for each token position sums to 1
np.sum(new_predicted_prob, axis = 2)

In [ ]:
# we just need the last word (in our sentence of 61 tokens), so we subset the last word in the array (second index)
# there is only one line, so the first index has to be index 0
# we want all the probabilities for the possible token in this last word position, so the last index is : (everything)
# recall that the shape was (1,9,3081)
new_predicted_prob_last = new_predicted_prob[0, -1, :]
new_predicted_prob_last

In [ ]:
new_predicted_prob_last.shape
# we see that we have extracted the 3081 probabilities for each word in our vocab for the last token position

In [ ]:
# now we make use of our new probability distribution. How? let's go back to the slides
# we sample one time for the last word, based on the new probability distribution
predicted = np.random.multinomial(1, new_predicted_prob_last, 1)
predicted

In [ ]:
# there should only be one word from the above that has occured once
# we find that word using argmax
predicted_word = np.argmax(predicted)
predicted_word

In [ ]:
# let's look up what this word is
for word, i in tokenizer.word_index.items():
    if i == predicted_word: 
        print(word)
        break

### So that's in general how we can inject some stochasticity into our word selection

In [ ]:
# we define a function for the temperature-based word selection to make the code 
# easier to read later for text generation
# it's just all the same steps above, but defined in a function "sample_next"

# this function takes the raw predicted probabilities of each word in our vocab, and the temp as inputs 
def sample_next(predicted_prob, temperature):
    predicted_prob = np.asarray(predicted_prob).astype('float64')
    predicted_prob = np.log(predicted_prob) / temperature

    exp_predicted_prob = np.exp(predicted_prob)
    new_predicted_prob = exp_predicted_prob / exp_predicted_prob.sum(axis = 2, keepdims = 1)
    
    new_predicted_prob_last = new_predicted_prob[0, -1,:]
    
    predicted = np.random.multinomial(1, new_predicted_prob_last, 1)
    
    return np.argmax(predicted)

In [ ]:
# Let's start generating text from our model!
# we begin with a seed text
start_text = "The flowers bloom"

# we generate 100 more words
next_words = 100

# repeat for the next 100 words 
for _ in range(next_words):
    # we tokenize the seed text or the text up till this point, and pad it
    token_list = tokenizer.texts_to_sequences([start_text])[0]
    token_list = pad_sequences([token_list], maxlen = max_sequence_length - 1, padding = 'pre')
    
    # get the softmax output   
    predicted_prob = model.predict(token_list)
    # modify the softmax output by the temperature, and sample the next word based on the new prob distr
    predicted_word = sample_next(predicted_prob, 1.5)
    
    output_word = ''
    
    # check what the predicted word is and add it to the seed text
    for word, i in tokenizer.word_index.items():
        if i == predicted_word: 
            output_word = word
            break
    start_text += " " + output_word
    
    # repeat for the next word position: sample stochastically and add to existing text
    
print(start_text)

#### What do we think of these results? Let's go back to the slides